# Image to 3D mesh on a free Colab T4 (Hunyuan3D-2mini)
Runtime -> Change runtime type -> **T4 GPU**, then run the cells in order. Upload your own picture in cell 3, or use the Hunyuan3D-2 demo picture.

In [ ]:
# 1. Install. --no-deps keeps Colab's own torch. Hunyuan3D's checkpoint uses transformers 4 layer names,
# so transformers is pinned below 5, and diffusers to a release that works with the matching huggingface-hub.
!pip -q install hy3dgen --no-deps
!pip -q install omegaconf einops trimesh pygltflib pyyaml safetensors opencv-python-headless rembg onnxruntime scikit-image pymeshlab fast-simplification
!pip -q install 'transformers>=4.49,<5' 'diffusers>=0.33,<0.36'

In [ ]:
# 2. Restart the Python process so numpy/scipy reload cleanly after the installs above.
# Colab will say the session crashed. That is expected. Continue with cell 3.
import os; os.kill(os.getpid(), 9)

In [ ]:
# 3. Pick an input picture: upload one, or fall back to the demo picture from the Hunyuan3D-2 repo.
from google.colab import files
IMAGE = None
try:
    up = files.upload()
    IMAGE = next(iter(up), None)
except Exception:
    pass
if not IMAGE:
    IMAGE = 'input_demo.png'
    !wget -q https://raw.githubusercontent.com/Tencent-Hunyuan/Hunyuan3D-2/main/assets/demo.png -O input_demo.png
print('Using', IMAGE)

In [ ]:
# 4. Generate the shape (turbo mini model, about 5 GB VRAM) and save a .glb
import time, torch
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini-turbo',
    use_safetensors=True, device='cuda')
img = Image.open(IMAGE).convert('RGBA')
if img.getextrema()[3][0] == 255:  # no transparency, so cut out the background
    from hy3dgen.rembg import BackgroundRemover
    img = BackgroundRemover()(img)
t = time.time()
mesh = pipe(image=img, num_inference_steps=5, octree_resolution=320, num_chunks=20000,
            generator=torch.manual_seed(12345), output_type='trimesh')[0]
mesh = mesh.simplify_quadric_decimation(face_count=30000)  # the raw mesh is ~500k faces; 30k is plenty for a game
print(f'done in {time.time()-t:.1f}s, peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
mesh.export('model.glb')

In [ ]:
# 5. Download it. Open the .glb in Blender, Godot, three.js, or https://gltf-viewer.donmccurdy.com
from google.colab import files
files.download('model.glb')